### Dokümanlar
- genel/sube_randevu.md
- hesap/hesap_kapatma.md
- kredi/kredi_belge_talebi.md

In [4]:
K =3
SKOR_ESIGI = 0.5
# Proje kökünü bul ve src/ klasörünü import yoluna ekle (Hafta 2 notebook'larındaki gibi)
import json
import sys
from pathlib import Path

# Notebook hangi klasörden açılırsa açılsın, kontrol.py'nin bulunduğu klasör proje köküdür
PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))  # artık "from asistan ..." çalışır

import numpy as np  # vektör hesapları (kosinüs benzerliği) için

# ayar: .env'den değer okur | embedding_getir: bge-m3 modeli | model_getir: sohbet modeli (yerel ya da bulut)
from asistan.ayarlar import ayar, embedding_getir, model_getir

MODEL = "bulut"   # canlı gösterimde "bulut" (Claude) daha tutarlı sonuç verir
EMBED = embedding_getir()  # indeksi kuran ve ajanın aramada kullandığı modelin aynısı
print("Embedding modeli:", ayar("EMBED_MODEL"))

Embedding modeli: bge-m3


In [18]:
from asistan.rag.arama import K, SKOR_ESIGI, ara  # ara(): prosedur_ara aracının arka planda çağırdığı fonksiyon

print(f"Varsayılan k = {K}, skor eşiği = {SKOR_ESIGI}\n")


def goster_parcalar(sorgu, kategoriler, k=K):  # ara() sonucunu yazdırır; model çağrılmaz
    print(f"Sorgu: {sorgu!r}  kategoriler={kategoriler}  k={k}")
    for p in ara(sorgu, kategoriler, k):
        durum = "✓" if p["skor"] >= SKOR_ESIGI else "✗ eşik altı"  # ✗ olan parçayı prosedur_ara modele vermez
        print(f"  {p['skor']:.2f} {durum:<12} {p['kaynak']}")
    print()

Varsayılan k = 3, skor eşiği = 0.5



### Yeni dokümanları indexleme den önce ilgili soruları mevcut dokümanlardan cevaplama

In [13]:
# T36'nin sorusu, hesap islemlerinden hesap kapatma alanı kaldırıldı
goster_parcalar("Uygulamadan hesabımı nasıl kapatırım", ["hesap"])

Sorgu: 'Uygulamadan hesabımı nasıl kapatırım'  kategoriler=['hesap']  k=3
  0.60 ✓            hesap/hesap_islemleri.md#Hesap kapama
  0.54 ✓            hesap/isletim_ucreti.md#Ücret iadesi
  0.54 ✓            hesap/dijital_kanallar.md#Sık görülen hatalar



In [14]:
# T37
goster_parcalar("Şubeye gitmeden önce nasıl randevu alırım?", ["genel"])

Sorgu: 'Şubeye gitmeden önce nasıl randevu alırım?'  kategoriler=['genel']  k=3



In [15]:
# T38
goster_parcalar("Kredi borcumun ne kadar kaldığını gösteren belgeyi nasıl alırım", ["kredi"])

Sorgu: 'Kredi borcumun ne kadar kaldığını gösteren belgeyi nasıl alırım'  kategoriler=['kredi']  k=3
  0.60 ✓            kredi/basvuru_sureci.md#İstenen belgeler
  0.57 ✓            kredi/basvuru_sureci.md#Değerlendirme
  0.55 ✓            kredi/erken_kapama.md#Tutarı öğrenme ve talimat



## Indexing

**Komut:**
```bash
python -m asistan.rag.indeksle --chunk 400 --ortusme 50
```

**Genel Bilgiler:**
- Toplam indekslenen parça: 69
- Chunk boyutu: 400
- Örtüşme: 50
- Embedding modeli: bge-m3
- İndeks dizini: `data/indeks/`

**İndekslenen Prosedürler:**

- **Kart**
  - `ekstre_ve_son_odeme.md` — 3 parça
  - `harcama_itirazi.md` — 4 parça
  - `ic_itiraz_degerlendirme_kriterleri.md` — 3 parça (ic)
  - `kart_aidati.md` — 3 parça
  - `kart_calismiyor_ve_limit.md` — 4 parça
  - `kayip_calinti_ve_dolandiricilik.md` — 4 parça

- **Hesap**
  - `dijital_kanallar.md` — 4 parça
  - `dolandiricilik_guvenli_hesap.md` — 3 parça
  - `hesap_islemleri.md` — 2 parça
  - `hesap_kapatma.md` — 5 parça
  - `isletim_ucreti.md` — 4 parça
  - `para_transferi.md` — 3 parça
  - `sube_randevu.md` — 4 parça
  - `ucret_itirazi.md` — 3 parça
  - `vadeli_mevduat.md` — 3 parça

- **Kredi**
  - `basvuru_sureci.md` — 3 parça
  - `erken_kapama.md` — 2 parça
  - `faiz_oranlari.md` — 2 parça
  - `gecikme_ve_yapilandirma.md` — 3 parça
  - `kefil_ve_teminat.md` — 2 parça
  - `kredi_belge_talebi.md` — 5 parça

In [19]:
# T36'nin sorusu, hesap işlemlerinden hesap kapatma alanı kaldırıldı, alakalı yeni prosedür eklendi
goster_parcalar("Uygulamadan hesabımı nasıl kapatırım", ["hesap"])

Sorgu: 'Uygulamadan hesabımı nasıl kapatırım'  kategoriler=['hesap']  k=3
  0.60 ✓            hesap/hesap_islemleri.md#Hesap kapama
  0.54 ✓            hesap/isletim_ucreti.md#Ücret iadesi
  0.54 ✓            hesap/dijital_kanallar.md#Sık görülen hatalar



In [20]:
# T37
goster_parcalar("Şubeye gitmeden önce nasıl randevu alırım?", ["hesap"])

Sorgu: 'Şubeye gitmeden önce nasıl randevu alırım?'  kategoriler=['hesap']  k=3
  0.55 ✓            hesap/dijital_kanallar.md#Şifre unutma ve yenileme
  0.54 ✓            hesap/dijital_kanallar.md#Şifre unutma ve yenileme
  0.54 ✓            hesap/dijital_kanallar.md#Biyometrik giriş



In [21]:
# T38
goster_parcalar("Kredi borcumun ne kadar kaldığını gösteren belgeyi nasıl alırım", ["kredi"])

Sorgu: 'Kredi borcumun ne kadar kaldığını gösteren belgeyi nasıl alırım'  kategoriler=['kredi']  k=3
  0.60 ✓            kredi/basvuru_sureci.md#İstenen belgeler
  0.57 ✓            kredi/basvuru_sureci.md#Değerlendirme
  0.55 ✓            kredi/erken_kapama.md#Tutarı öğrenme ve talimat



In [23]:
from asistan import cevapla  # projenin ortak arayüzü: sınıflandır → ajan → taslak → kontroller


def goster(sonuc):  # cevapla() sonucunu okunur biçimde yazdırır
    print("Adımlar:")
    for adim in sonuc["adimlar"]:
        print("  -", adim)
    print("\nDavranış  :", sonuc["davranis"])
    print("Araçlar   :", ", ".join(sonuc["kullanilan_araclar"]) or "-")
    print("Kaynaklar :", sonuc["kaynaklar"] or "-")
    print("Yanıt     :", sonuc["yanit"])


# M0001
goster(cevapla("Uygulamadan hesabımı nasıl kapatırım?", "M0001", MODEL))

Adımlar:
  - talep: Uygulamadan hesabımı nasıl kapatırım?
  - siniflandirma: hesap (ikinci: -, guven: 0.95) - Müşteri mobil uygulama üzerinden hesabını kapatmanın yolunu soruyor, bu hesap kapama konusudur.
  - arac seti (hesap): musteri_bilgisi, hesap_durumu, gecmis_talepler, prosedur_ara, doviz_kuru
  - ajan adim 1: prosedur_ara({'sorgu': 'hesap kapatma uygulamadan nasıl yapılır'})
  - prosedur_ara: hesap/hesap_islemleri.md#Hesap kapama (0.67), hesap/hesap_islemleri.md#IBAN öğrenme (0.56), hesap/ucret_itirazi.md#İtiraz nasıl yapılır (0.55)
  - ajan adim 2: hesap_durumu({})
  - ajan adim 3: arac cagrisi yok, bulgular yazildi
  - taslak: temsilciye_aktar - Hesap kapama insan onayı gerektiren bir yazma işlemi olduğundan asistan yapamaz ve talep temsilciye aktarılmalıdır. | baglam: 3 parça, kaynaklar: hesap/hesap_islemleri.md#Hesap kapama
  - risk kontrolu: ajan temsilciye aktarmayi secti
  - temsilciye aktarildi: ajan kararı: Hesap kapama insan onayı gerektiren bir yazma işlemi olduğunda

In [24]:
# M0001
goster(cevapla("Şubeye gitmeden önce nasıl randevu alırım", "M0001", MODEL))

Adımlar:
  - talep: Şubeye gitmeden önce nasıl randevu alırım
  - siniflandirma: diger (ikinci: -, guven: 0.60) - Şube randevusu almak belirli bir bankacılık ürününe (kart, hesap, kredi) bağlanamayan genel bir istektir.
  - hazir yanit: kapsam_disi

Davranış  : kapsam_disi
Araçlar   : -
Kaynaklar : -
Yanıt     : Bu konu Örnek Bank müşteri asistanının kapsamı dışında kalıyor. Kart, hesap ve kredi işlemlerinizle ilgili sorularınızda yardımcı olabilirim; bir temsilciyle görüşmek isterseniz 0850 000 00 00 numaralı hattımızı arayabilirsiniz.


In [25]:
# M0001
goster(cevapla("Kredi borcumun ne kadar kaldığını gösteren belgeyi nasıl alırım", "M0001", MODEL))

Adımlar:
  - talep: Kredi borcumun ne kadar kaldığını gösteren belgeyi nasıl alırım
  - siniflandirma: kredi (ikinci: -, guven: 0.90) - Müşteri kalan kredi borcunu gösteren belgeyi sorduğu için talep kredi durumuyla ilgilidir ve başka bir konu içermez.
  - arac seti (kredi): musteri_bilgisi, kredi_basvuru_durumu, gecmis_talepler, prosedur_ara
  - ajan adim 1: prosedur_ara({'sorgu': 'kredi borç durum belgesi kalan borç yazısı nasıl alınır'}), kredi_basvuru_durumu({})
  - prosedur_ara: kredi/basvuru_sureci.md#Değerlendirme (0.57), kredi/basvuru_sureci.md#İstenen belgeler (0.54), kredi/basvuru_sureci.md#Başvuru kanalları (0.52)
  - ajan adim 2: prosedur_ara({'sorgu': 'borç dökümü kapama yazısı kalan taksit belgesi'})
  - prosedur_ara: kredi/gecikme_ve_yapilandirma.md#Gecikmiş taksit (0.55), kredi/erken_kapama.md#Tutarı öğrenme ve talimat (0.52), kredi/basvuru_sureci.md#Değerlendirme (0.51)
  - ajan adim 3: arac cagrisi yok, bulgular yazildi
  - taslak: temsilciye_aktar - Kredi borç durum 